In [0]:
import os
# Suppress TF C++ warnings (0=all, 1=no info, 2=no info/warnings)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import uuid

import keras
import keras_tuner as kt
import tensorflow as tf
import matplotlib.pyplot as plt

# Configure GPU memory growth to prevent pre-allocation
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

from learning_to_rank.data import build_dataset
from learning_to_rank.tuning.hypermodels import TransformerRankerHyperModel
from learning_to_rank.callbacks import TransformerWarmupCallback

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"Visible GPUs: {len(gpus)}")

In [0]:
CONFIG = {
    "project_name": "transformer_notebook_tuning",
    "train_path": "/dbfs/datasets/MSLR-WEB30K/Combined/train.tfrecord",
    "val_path": "/dbfs/datasets/MSLR-WEB30K/Combined/vali.tfrecord",
    "max_epochs": 20,
    "warmup_epochs": 10,
    "batch_size": 128,
    "num_features": 136,
    "model_id": None,
}

model_id = CONFIG["model_id"] or uuid.uuid4().hex

print(f"{model_id=}")

In [0]:
# with tf.device("cpu"):
train_ds = build_dataset(CONFIG["train_path"], batch_size=CONFIG["batch_size"], shuffle=True)
val_ds = build_dataset(CONFIG["val_path"], batch_size=CONFIG["batch_size"], shuffle=False)

# Peek at one batch
for x_batch, y_batch in train_ds.take(1):
    print(f"Input batch shape: {x_batch.shape}")  # [Batch, List, Features]
    print(f"Label batch shape: {y_batch.shape}")  # [Batch, List]

In [0]:
hypermodel = TransformerRankerHyperModel(num_features=CONFIG["num_features"])

tuner = kt.Hyperband(
    hypermodel,
    objective=kt.Objective("val_ndcg", direction="max"),
    max_epochs=CONFIG["max_epochs"],
    factor=1,
    directory=f"models/{model_id}/tuning",
    project_name=CONFIG["project_name"],
    executions_per_trial=1,  # Reduced to 1 to prevent parallel OOM
)

callbacks = [
    TransformerWarmupCallback(warmup_epochs=CONFIG["warmup_epochs"]),
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)
]

In [0]:
import gc

print(f"Starting Transformer Optimization...")
try:
    # Clear any existing models from memory
    keras.backend.clear_session()
    gc.collect()
    
    tuner.search(
        train_ds,
        validation_data=val_ds,
        callbacks=callbacks,
        verbose=1, # Changed to 1 for notebook progress bars
    )
except Exception as e:
    print(f"Error during tuning: {e}")
    import traceback
    traceback.print_exc()
    
    # Clean up on error
    keras.backend.clear_session()
    gc.collect()

In [0]:
# Display summary of results
tuner.results_summary()

In [0]:
# Retrieve best model
best_models = tuner.get_best_models(num_models=5)
for i, best_model in enumerate(best_models):
    best_model.save(f"models/transformer_{model_id}-{i}.keras")
    print(f"[SUCCESS] Best Transformer model {model_id}-{i} saved.")